# Figure 5-2: all-gene CellChat analysis

This notebook retains the CellChat analysis required for the Epi_JUN-CM1 top-25% circle plot and the Epi_JUN-to-CM1 ligand-receptor bubble plot.


## 1. Prepare the all-gene CellChat input


In [ ]:
from __future__ import annotations

from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
from scipy import io as spio


METHOD_DIR = Path.cwd() / "cellchat_r_all_genes"
INPUTS = METHOD_DIR / "inputs"
OUTPUTS = METHOD_DIR / "outputs"

INPUT_H5AD = INPUTS / "adata_anno_cell_subtype_re.h5ad"
SUBTYPE_INPUT_H5AD = INPUTS / "subtype_full_lr_input.h5ad"
SUBTYPE_PAIRS_CSV = INPUTS / "subtype_all_pairs.csv"
CM_NODE_MAPPING_CSV = INPUTS / "cm_node_mapping_for_subtype_aggregation.csv"
EPI_SUBTYPES_CSV = INPUTS / "epi_subtypes.csv"
GROUP_KEY = "ccc_group"
SUBTYPE_KEY = "cell_subtype"


def read_required_csv(path: Path, required_cols: set[str]) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {path}")
    frame = pd.read_csv(path)
    if not required_cols.issubset(frame.columns):
        raise RuntimeError(f"{path} must contain {sorted(required_cols)}")
    return frame


def load_cm_node_mapping(valid_subtypes: set[str]) -> pd.DataFrame:
    mapping = read_required_csv(
        CM_NODE_MAPPING_CSV,
        {"CM", "reference_node_rank", "node"},
    ).copy()
    mapping["CM"] = mapping["CM"].astype(str)
    mapping["node"] = mapping["node"].astype(str)
    mapping = mapping[mapping["CM"].str.startswith("joint_")]
    mapping = mapping.drop_duplicates(["CM", "node"])
    if "node_in_adata" in mapping.columns:
        marked = mapping["node_in_adata"].astype(str).str.lower().isin(["true", "1", "yes"])
        mapping = mapping[marked]
    return mapping[mapping["node"].isin(valid_subtypes)].copy()


def export_cellchat_input(adata_obj: ad.AnnData) -> None:
    output_dir = INPUTS / "cellchat_input"
    output_dir.mkdir(parents=True, exist_ok=True)
    spio.mmwrite(output_dir / "expression_genes_by_cells.mtx", adata_obj.X.T)
    pd.DataFrame({"gene": adata_obj.var_names}).to_csv(
        output_dir / "genes.tsv", sep="\t", index=False
    )
    pd.DataFrame({"cell": adata_obj.obs_names}).to_csv(
        output_dir / "cells.tsv", sep="\t", index=False
    )
    optional = [
        column
        for column in ["cell_type", "cell_subtype", "sample", "status"]
        if column in adata_obj.obs.columns
    ]
    metadata = adata_obj.obs[[GROUP_KEY] + optional].copy()
    metadata.insert(0, "cell", adata_obj.obs_names)
    metadata.to_csv(output_dir / "metadata.tsv", sep="\t", index=False)


def prepare_input(force: bool = False) -> None:
    required_exports = [
        SUBTYPE_INPUT_H5AD,
        INPUTS / "cellchat_input/expression_genes_by_cells.mtx",
        INPUTS / "cellchat_input/genes.tsv",
        INPUTS / "cellchat_input/cells.tsv",
        INPUTS / "cellchat_input/metadata.tsv",
    ]
    if all(path.exists() for path in required_exports) and not force:
        return
    if not INPUT_H5AD.exists():
        raise FileNotFoundError(f"Missing input AnnData: {INPUT_H5AD}")

    adata = ad.read_h5ad(INPUT_H5AD, backed="r")
    obs = adata.obs.copy()
    if SUBTYPE_KEY not in obs.columns:
        raise RuntimeError(f"{INPUT_H5AD} lacks obs['{SUBTYPE_KEY}']")
    obs[SUBTYPE_KEY] = obs[SUBTYPE_KEY].astype(str)

    all_subtypes = set(obs[SUBTYPE_KEY].unique())
    cm_mapping = load_cm_node_mapping(all_subtypes)
    cm_nodes = sorted(cm_mapping["node"].unique())
    epithelial = read_required_csv(EPI_SUBTYPES_CSV, {"Epi"})
    pair_table = read_required_csv(SUBTYPE_PAIRS_CSV, {"source", "target"})
    analysis_groups = sorted(
        (set(pair_table["source"].astype(str)) | set(pair_table["target"].astype(str)))
        & all_subtypes
    )
    selected_names = obs.index[obs[SUBTYPE_KEY].isin(analysis_groups)].tolist()

    if adata.raw is not None:
        result = adata.raw[selected_names, :].to_adata().copy()
        expression_source = ".raw normalized/log expression"
    else:
        result = adata[selected_names, :].to_memory().copy()
        expression_source = ".X expression"

    result.obs[GROUP_KEY] = result.obs[SUBTYPE_KEY].astype(str)
    result.obs["direction_class"] = np.where(
        result.obs[GROUP_KEY].str.startswith("Epi_"),
        "Epi",
        np.where(result.obs[GROUP_KEY].isin(cm_nodes), "CM_node", "Other_subtype"),
    )
    result.uns["cm_groups"] = sorted(cm_mapping["CM"].unique())
    result.uns["epi_groups"] = [
        name for name in epithelial["Epi"].astype(str) if name in all_subtypes
    ]
    result.uns["cm_nodes"] = cm_nodes
    result.uns["all_subtype_groups"] = analysis_groups
    result.uns["input_note"] = (
        "All-gene CellChat input without downsampling; grouped by cell_subtype; "
        f"expression source: {expression_source}."
    )
    result.write_h5ad(SUBTYPE_INPUT_H5AD)
    export_cellchat_input(result)

prepare_input(force=False)


## 2. Run CellChat and retain the two required analysis outputs


In [ ]:
%%script R
method_dir <- file.path(getwd(), "cellchat_r_all_genes")
input_dir <- file.path(method_dir, "inputs/cellchat_input")
output_dir <- file.path(method_dir, "outputs")
pairs_csv <- file.path(method_dir, "inputs/subtype_all_pairs.csv")
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
Sys.setenv(TMPDIR = "/mnt/disk18t/lr_xcy/riku/codex_research/envs/cm_epi_communication_r_tmp_official_ccc")
Sys.setenv(R_LIBS_USER = "/mnt/disk18t/lr_xcy/riku/codex_research/envs/r-cellchat")
.libPaths(c(Sys.getenv("R_LIBS_USER"), .libPaths()))
suppressPackageStartupMessages({
  library(CellChat)
  library(Matrix)
  library(future)
})

cellchat_workers <- as.integer(Sys.getenv("CELLCHAT_WORKERS", "8"))
if (is.na(cellchat_workers) || cellchat_workers < 1) {
  cellchat_workers <- 1
}
options(future.globals.maxSize = 100 * 1024^3)
future::plan("multisession", workers = cellchat_workers)

expr <- readMM(file.path(input_dir, "expression_genes_by_cells.mtx"))
genes <- read.delim(file.path(input_dir, "genes.tsv"), stringsAsFactors = FALSE)
cells <- read.delim(file.path(input_dir, "cells.tsv"), stringsAsFactors = FALSE)
meta <- read.delim(file.path(input_dir, "metadata.tsv"), stringsAsFactors = FALSE)
rownames(expr) <- make.unique(genes$gene)
colnames(expr) <- cells$cell
rownames(meta) <- meta$cell
meta$ccc_group <- factor(meta$ccc_group)

cellchat <- createCellChat(object = expr, meta = meta, group.by = "ccc_group")
cellchat@DB <- CellChatDB.human
cellchat <- subsetData(cellchat)
cellchat <- identifyOverExpressedGenes(cellchat)
cellchat <- identifyOverExpressedInteractions(cellchat)
cellchat <- computeCommunProb(cellchat, type = "triMean", nboot = 100, seed.use = 1337)
cellchat <- filterCommunication(cellchat, min.cells = 10)
cellchat <- computeCommunProbPathway(cellchat)
cellchat <- aggregateNet(cellchat)
saveRDS(cellchat, file.path(output_dir, "cellchat_object.rds"))

all_communication <- subsetCommunication(cellchat)
pairs <- read.csv(pairs_csv, stringsAsFactors = FALSE)
selected_communication <- merge(
  all_communication,
  pairs[, c("source", "target")],
  by = c("source", "target")
)
write.csv(
  selected_communication,
  file.path(output_dir, "official_cellchat_subtype_lr_results.csv"),
  row.names = FALSE
)

future::plan("sequential")


## 3. Prepare the Epi_JUN-to-CM1 bubble-plot CSV


In [ ]:

METHOD_DIR = Path.cwd() / "cellchat_r_all_genes"
OUTPUT_DIR = METHOD_DIR / "outputs"
MAPPING_CSV = METHOD_DIR / "inputs/cm_node_mapping_for_subtype_aggregation.csv"
CELLCHAT_LR_CSV = OUTPUT_DIR / "official_cellchat_subtype_lr_results.csv"
BUBBLE_DIR = OUTPUT_DIR / "claim_support/epi_jun_cm1_bubble"
BUBBLE_CSV = BUBBLE_DIR / "cellchat_official_epi_jun_cm1_bubble_data.csv"

SOURCE = "Epi_JUN"
CM = "joint_01_sharedCM"
KEEP_LR_PAIRS = {
    "GDF15|TGFBR2",
    "MIF|CD74_CD44",
    "MIF|CD74_CXCR4",
    "SPP1|CD44",
    "SPP1|ITGA4_ITGB1",
    "SPP1|ITGAV_ITGB1",
    "SPP1|ITGAV_ITGB5",
    "TGFB1|TGFbR1_R2",
    "TNFSF9|TNFRSF9",
}


def load_cm_nodes() -> list[str]:
    mapping = pd.read_csv(MAPPING_CSV)
    if "node_in_adata" in mapping.columns:
        keep = mapping["node_in_adata"].astype(str).str.lower().isin(["true", "t", "1", "yes"])
        mapping = mapping.loc[keep].copy()
    selected = mapping.loc[mapping["CM"].astype(str).eq(CM)].copy()
    selected = selected.sort_values("reference_node_rank")
    return selected["node"].astype(str).drop_duplicates().tolist()


def make_epi_jun_cm1_bubble_data() -> None:
    communication = pd.read_csv(CELLCHAT_LR_CSV)
    communication["source"] = communication["source"].astype(str)
    communication["target"] = communication["target"].astype(str)
    communication["ligand"] = communication["ligand"].astype(str)
    communication["receptor"] = communication["receptor"].astype(str)
    communication["pval"] = pd.to_numeric(communication["pval"], errors="coerce")
    communication["communication_weight"] = pd.to_numeric(
        communication["communication_weight"]
        if "communication_weight" in communication.columns
        else communication["prob"],
        errors="coerce",
    ).fillna(0)
    communication["lr_pair"] = communication["ligand"] + "|" + communication["receptor"]

    bubble_data = communication.loc[
        communication["source"].eq(SOURCE)
        & communication["target"].isin(load_cm_nodes())
        & communication["pval"].le(0.05)
        & communication["annotation"].astype(str).eq("Secreted Signaling")
        & communication["lr_pair"].isin(KEEP_LR_PAIRS)
    ].copy()
    bubble_data = bubble_data.sort_values(["target", "lr_pair"]).reset_index(drop=True)
    BUBBLE_DIR.mkdir(parents=True, exist_ok=True)
    bubble_data.to_csv(BUBBLE_CSV, index=False)


make_epi_jun_cm1_bubble_data()


## 4. Draw the retained figures


In [ ]:
%%script R
method_dir <- file.path(getwd(), "cellchat_r_all_genes")
task_dir <- file.path(method_dir, "official_circle_top_joint01_epi_jun")
source_output_dir <- file.path(method_dir, "outputs")
result_dir <- file.path(task_dir, "outputs")
cm_name <- "joint_01_sharedCM"
epi_name <- "Epi_JUN"
top_fraction <- 0.25
Sys.setenv(R_LIBS_USER = "/mnt/disk18t/lr_xcy/riku/codex_research/envs/r-cellchat")
.libPaths(c(Sys.getenv("R_LIBS_USER"), .libPaths()))
suppressPackageStartupMessages(library(CellChat))

cellchat_path <- file.path(source_output_dir, "cellchat_object.rds")
mapping_path <- file.path(method_dir, "inputs/cm_node_mapping_for_subtype_aggregation.csv")
group_counts_path <- file.path(method_dir, "inputs/group_counts.csv")
for (path in c(cellchat_path, mapping_path, group_counts_path)) {
  if (!file.exists(path)) {
    stop("Missing required input: ", path)
  }
}

cellchat <- readRDS(cellchat_path)
mapping <- read.csv(mapping_path, stringsAsFactors = FALSE)
group_counts <- read.csv(group_counts_path, stringsAsFactors = FALSE)
if ("node_in_adata" %in% colnames(mapping)) {
  keep <- tolower(as.character(mapping$node_in_adata)) %in% c("true", "t", "1", "yes")
  mapping <- mapping[keep, , drop = FALSE]
}

cm_mapping <- mapping[mapping$CM == cm_name, , drop = FALSE]
cm_mapping <- cm_mapping[order(cm_mapping$reference_node_rank), , drop = FALSE]
cm_nodes <- unique(as.character(cm_mapping$node))
nodes <- unique(c(epi_name, cm_nodes))
nodes <- nodes[nodes %in% levels(cellchat@idents)]
if (length(nodes) < 2) {
  stop("Fewer than two valid nodes for ", cm_name, " and ", epi_name)
}

count_matrix <- cellchat@net$count[nodes, nodes, drop = FALSE]
weight_matrix <- cellchat@net$weight[nodes, nodes, drop = FALSE]
group_size <- as.numeric(table(cellchat@idents))
names(group_size) <- names(table(cellchat@idents))
if (all(c("group", "used_cells") %in% colnames(group_counts))) {
  used_cells <- as.numeric(group_counts$used_cells)
  names(used_cells) <- group_counts$group
  matched <- names(group_size) %in% names(used_cells)
  group_size[matched] <- used_cells[names(group_size)[matched]]
}
valid_group_size <- group_size[nodes]
valid_group_size[is.na(valid_group_size)] <- 1

dir.create(result_dir, recursive = TRUE, showWarnings = FALSE)
output_pdf <- file.path(
  result_dir,
  "joint_01_sharedCM__Epi_JUN__netVisual_circle_official_top25.pdf"
)
pdf(output_pdf, width = 14, height = 7, useDingbats = FALSE)
par(mfrow = c(1, 2), xpd = TRUE)
netVisual_circle(
  count_matrix,
  vertex.weight = valid_group_size,
  top = top_fraction,
  weight.scale = TRUE,
  label.edge = FALSE,
  title.name = "Number of interactions: Epi_JUN - joint_01_sharedCM (official top=25%)",
  remove.isolate = FALSE
)
netVisual_circle(
  weight_matrix,
  vertex.weight = valid_group_size,
  top = top_fraction,
  weight.scale = TRUE,
  label.edge = FALSE,
  title.name = "Interaction weights/strength: Epi_JUN - joint_01_sharedCM (official top=25%)",
  remove.isolate = FALSE
)
dev.off()

filter_stats <- function(net, metric) {
  threshold <- as.numeric(stats::quantile(net, probs = 1 - top_fraction, names = FALSE))
  filtered <- net
  filtered[filtered < threshold] <- 0
  positive <- filtered > 0
  data.frame(
    metric = metric,
    top_fraction = top_fraction,
    quantile_probability = 1 - top_fraction,
    threshold = threshold,
    matrix_entries = length(net),
    original_nonzero_edges = sum(net > 0, na.rm = TRUE),
    retained_nonzero_edges = sum(positive, na.rm = TRUE),
    retained_self_edges = sum(diag(positive), na.rm = TRUE),
    retained_nonself_edges = sum(positive, na.rm = TRUE) - sum(diag(positive), na.rm = TRUE),
    remove_isolate = FALSE,
    stringsAsFactors = FALSE
  )
}

audit <- rbind(
  filter_stats(count_matrix, "count"),
  filter_stats(weight_matrix, "weight")
)
audit$filter_definition <- "CellChat official: threshold=quantile(net, 1-top); net[net<threshold]=0"
write.csv(audit, file.path(result_dir, "official_top_filter_audit.csv"), row.names = FALSE)


In [ ]:
import matplotlib

matplotlib.use("Agg")
matplotlib.rcParams["pdf.fonttype"] = 42
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


METHOD_DIR = Path.cwd() / "cellchat_r_all_genes"
BUBBLE_DIR = METHOD_DIR / "outputs/claim_support/epi_jun_cm1_bubble"
DATA_CSV = BUBBLE_DIR / "cellchat_official_epi_jun_cm1_bubble_data.csv"
OUTPUT_PDF = BUBBLE_DIR / "cellchat_official_epi_jun_cm1_bubble_plot.pdf"


def plot_epi_jun_cm1_bubble() -> None:
    data = pd.read_csv(DATA_CSV)
    color_col = "prob" if "prob" in data.columns else "lr_probs"
    size_col = "communication_weight"
    data[color_col] = pd.to_numeric(data[color_col], errors="coerce").fillna(0)
    data[size_col] = pd.to_numeric(data[size_col], errors="coerce").fillna(0)
    data["lr_pair"] = data["lr_pair"].astype(str)
    data["target"] = data["target"].astype(str)

    target_order = data.groupby("target", observed=True)[size_col].sum().sort_values(ascending=False).index.tolist()
    pair_order = data.groupby("lr_pair", observed=True)[size_col].max().sort_values(ascending=True).index.tolist()
    x_map = {value: index for index, value in enumerate(target_order)}
    y_map = {value: index for index, value in enumerate(pair_order)}

    weights = data[size_col].to_numpy(dtype=float)
    max_weight = np.nanmax(weights) if len(weights) else 0
    sizes = 18 + 210 * np.sqrt(weights / max_weight) if max_weight > 0 else np.repeat(28, len(data))
    fig_width = min(max(7.5, 0.55 * len(target_order) + 3.0), 11.0)
    fig_height = min(max(4.35, 0.20 * len(pair_order) + 2.1), 7.6)

    fig, axis = plt.subplots(figsize=(fig_width, fig_height))
    scatter = axis.scatter(
        data["target"].map(x_map).to_numpy(),
        data["lr_pair"].map(y_map).to_numpy(),
        c=data[color_col].to_numpy(dtype=float),
        s=sizes,
        cmap="viridis",
        edgecolors="none",
        linewidths=0,
        alpha=0.9,
    )
    axis.set_xticks(range(len(target_order)))
    axis.set_xticklabels(target_order, rotation=45, ha="right", fontsize=7)
    axis.set_yticks(range(len(pair_order)))
    axis.set_yticklabels(pair_order, fontsize=6)
    axis.set_xlabel("CM component subtype", fontsize=8)
    axis.set_ylabel("Ligand-receptor pair", fontsize=8)
    axis.set_title("CellChat official: Epi_JUN to CM1", fontsize=10)
    axis.grid(True, axis="both", color="0.88", linewidth=0.5)
    axis.set_axisbelow(True)
    colorbar = fig.colorbar(scatter, ax=axis, fraction=0.025, pad=0.01)
    colorbar.set_label(color_col, fontsize=7)
    colorbar.ax.tick_params(labelsize=6)

    if np.any(weights > 0):
        legend_values = np.quantile(weights[weights > 0], [0.33, 0.66, 1.0])
        handles = [
            plt.scatter(
                [], [], s=18 + 210 * np.sqrt(value / max_weight),
                color="0.55", edgecolors="none", linewidths=0,
            )
            for value in legend_values
        ]
        axis.legend(
            handles,
            [f"{value:.3f}" for value in legend_values],
            title=size_col,
            loc="upper right",
            bbox_to_anchor=(1.0, 1.02),
            fontsize=6,
            title_fontsize=6,
            frameon=True,
        )
    fig.tight_layout()
    fig.savefig(OUTPUT_PDF, bbox_inches="tight")
    plt.close(fig)


plot_epi_jun_cm1_bubble()
